In [ ]:
import json
import random
from pathlib import Path

from datasets import load_dataset


def has_valid_erqa_fields(ex):
    """
    ERQA rows should contain:
      - question_id
      - question
      - answer
      - images (list of 1+ images)
    """
    question_id = ex.get("question_id")
    question = ex.get("question")
    answer = ex.get("answer")
    images = ex.get("images")

    return (
        isinstance(question_id, str)
        and question_id.strip() != ""
        and isinstance(question, str)
        and question.strip() != ""
        and isinstance(answer, str)
        and answer.strip().upper() in {"A", "B", "C", "D"}
        and images is not None
        and isinstance(images, (list, tuple))
        and len(images) >= 1
    )


def sample_erqa_to_jsonl(
    output_jsonl="erqa_sample_400.jsonl",
    num_samples=400,
    seed=42,
    image_output_dir="erqa_sample_images",
    dataset_name="FlagEval/ERQA",
    split="test",
):
    """
    Sample random distinct examples from ERQA and save as JSONL.

    Output format per line:
    {
        "id": "...",
        "image_paths": ["...", "..."],
        "prompt": "..."
    }

    Notes:
    - ERQA questions already include the choices and the answer-format instruction,
      so we use ex["question"] directly as the prompt.
    - Each example may contain multiple images.
    """
    random.seed(seed)

    ds = load_dataset(dataset_name, split=split)

    valid_indices = [i for i in range(len(ds)) if has_valid_erqa_fields(ds[i])]

    if num_samples > len(valid_indices):
        raise ValueError(
            f"Requested {num_samples} samples, but only {len(valid_indices)} valid rows "
            f"were found in split '{split}'."
        )

    sampled_indices = random.sample(valid_indices, num_samples)

    image_output_dir = Path(image_output_dir)
    image_output_dir.mkdir(parents=True, exist_ok=True)

    output_path = Path(output_jsonl)

    with output_path.open("w", encoding="utf-8") as f:
        for _, idx in enumerate(sampled_indices, start=1):
            ex = ds[idx]

            question_id = ex["question_id"]
            question = ex["question"]
            answer_letter = str(ex["answer"]).strip().upper()
            images = ex["images"]

            # Make a folder per example, since each example may have multiple images
            example_dir = image_output_dir / question_id
            example_dir.mkdir(parents=True, exist_ok=True)

            image_paths = []
            for img_idx, img in enumerate(images, start=1):
                local_image_name = f"img{img_idx}.png"
                local_image_path = example_dir / local_image_name
                img.save(local_image_path)
                image_paths.append(str(local_image_path))

            # Traceable id
            example_id = f"row{idx}__qid{question_id}__ans{answer_letter}"

            record = {
                "id": example_id,
                "image_paths": image_paths,
                "prompt": question,
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Loaded split '{split}' from {dataset_name}")
    print(f"Found {len(valid_indices)} valid rows")
    print(f"Saved {num_samples} samples to {output_path}")
    print(f"Saved images under {image_output_dir}")


if __name__ == "__main__":
    sample_erqa_to_jsonl()

In [ ]:
import json
import re
import string
from pathlib import Path
from collections import Counter

from datasets import load_dataset


def normalize_text(s: str):
    if s is None:
        return None
    return " ".join(str(s).strip().lower().split())


def extract_option_letter(text: str, num_choices: int = None):
    """
    Try to parse the model output as a letter choice.

    Supports outputs such as:
      A
      a
      A)
      A.
      (A)
      Answer: A
      The answer is B

    Returns uppercase letter if found, else None.

    If num_choices is given, only returns letters within the valid range.
    """
    if text is None:
        return None

    text = text.strip()

    valid_letters = string.ascii_uppercase
    if num_choices is not None:
        valid_letters = valid_letters[:num_choices]

    # strict direct one-letter style
    m = re.fullmatch(r"\s*[\(\[]?([A-Za-z])[\)\]\.\:]?\s*", text)
    if m:
        letter = m.group(1).upper()
        if letter in valid_letters:
            return letter

    # common answer patterns
    patterns = [
        r"\banswer\s*[:\-]?\s*([A-Za-z])\b",
        r"\boption\s*[:\-]?\s*([A-Za-z])\b",
        r"\bchoice\s*[:\-]?\s*([A-Za-z])\b",
        r"\bthe answer is\s*([A-Za-z])\b",
        r"\b([A-Za-z])[\)\.]\s*",
    ]

    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            letter = m.group(1).upper()
            if letter in valid_letters:
                return letter

    # fallback: first standalone single-letter token
    for m in re.finditer(r"\b([A-Za-z])\b", text):
        letter = m.group(1).upper()
        if letter in valid_letters:
            return letter

    return None


def parse_choices_from_question(question: str):
    """
    ERQA embeds the choices in the question text, usually like:
      Choices: A. ... B. ... C. ... D. ... Please answer directly ...

    Returns a list of choice texts in A/B/C/D order when parsing succeeds.
    Otherwise returns None.
    """
    if not isinstance(question, str) or "Choices:" not in question:
        return None

    # keep only the part after "Choices:"
    try:
        tail = question.split("Choices:", 1)[1]
    except Exception:
        return None

    # cut off instruction text if present
    stop_markers = [
        "Please answer directly",
        "Please answer with",
        "Answer directly",
        "Answer with",
    ]
    for marker in stop_markers:
        if marker in tail:
            tail = tail.split(marker, 1)[0]

    tail = tail.strip()

    # Extract blocks for A/B/C/D
    # This captures text after A. until B., after B. until C., etc.
    matches = list(
        re.finditer(
            r"([A-D])\.\s*(.*?)(?=(?:\s+[A-D]\.\s)|$)",
            tail,
            flags=re.DOTALL,
        )
    )

    if not matches:
        return None

    found = {}
    for m in matches:
        letter = m.group(1).upper()
        text = m.group(2).strip()
        found[letter] = text

    # return in alphabetical order if contiguous from A
    choices = []
    for letter in ["A", "B", "C", "D"]:
        if letter in found:
            choices.append(found[letter])

    return choices if len(choices) >= 2 else None


def extract_choice_by_text(text: str, choices):
    """
    Fallback parser:
    if the model outputs the actual choice text instead of the letter,
    try to match it against one of the choices.

    Returns the matched choice index, else None.
    """
    if text is None:
        return None

    norm_pred = normalize_text(text)
    if not norm_pred:
        return None

    normalized_choices = [normalize_text(c) for c in choices]

    # strict exact match
    for i, c in enumerate(normalized_choices):
        if norm_pred == c:
            return i

    # substring matching
    for i, c in enumerate(normalized_choices):
        if c and c in norm_pred:
            return i

    return None


def has_valid_erqa_fields(ex):
    question_id = ex.get("question_id")
    question = ex.get("question")
    answer = ex.get("answer")
    images = ex.get("images")

    return (
        isinstance(question_id, str)
        and question_id.strip() != ""
        and isinstance(question, str)
        and question.strip() != ""
        and isinstance(answer, str)
        and answer.strip().upper() in {"A", "B", "C", "D"}
        and images is not None
        and isinstance(images, (list, tuple))
        and len(images) >= 1
    )


def build_gt_index(dataset_name="FlagEval/ERQA", split="test"):
    """
    Build a mapping from the custom example_id format used in the ERQA prep script
    to the ground-truth information.

    Expected example_id format:
      row123__qidERQA_20__ansB
    """
    ds = load_dataset(dataset_name, split=split)

    gt_index = {}

    for row_idx, ex in enumerate(ds):
        if not has_valid_erqa_fields(ex):
            continue

        question_id = ex["question_id"]
        question = ex["question"]
        question_type = ex.get("question_type")
        gt_letter = str(ex["answer"]).strip().upper()
        gt_index_num = string.ascii_uppercase.index(gt_letter)

        choices = parse_choices_from_question(question)
        gt_answer_text = None
        if choices is not None and 0 <= gt_index_num < len(choices):
            gt_answer_text = choices[gt_index_num]

        example_id = f"row{row_idx}__qid{question_id}__ans{gt_letter}"

        gt_index[example_id] = {
            "row_index": row_idx,
            "question_id": question_id,
            "question": question,
            "question_type": question_type,
            "choices": choices,
            "gt_answer_index": gt_index_num,
            "gt_answer_letter": gt_letter,
            "gt_answer_text": gt_answer_text,
        }

    return gt_index


def evaluate_result_folder(
    results_dir,
    output_json_path="evaluation_results_erqa.json",
    dataset_name="FlagEval/ERQA",
    split="test",
):
    results_dir = Path(results_dir)
    json_files = sorted(results_dir.glob("*.json"))

    if not json_files:
        raise ValueError(f"No JSON files found in: {results_dir}")

    gt_index = build_gt_index(dataset_name=dataset_name, split=split)

    per_example = {}
    stats = Counter()

    for json_file in json_files:
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            per_example[json_file.name] = {
                "error": f"Could not read JSON: {e}",
                "correct": None,
            }
            stats["read_error"] += 1
            continue

        example_id = data.get("example", {}).get("example_id")
        raw_answer = data.get("chat_result", {}).get("answer")

        # fallback to full_text if answer missing
        if raw_answer is None:
            raw_answer = data.get("chat_result", {}).get("full_text")

        if example_id is None:
            per_example[json_file.name] = {
                "example_id": None,
                "pred_answer_raw": raw_answer,
                "pred_answer_letter": None,
                "pred_answer_index": None,
                "gt_answer_index": None,
                "gt_answer_letter": None,
                "correct": None,
                "error": "Missing example.example_id",
            }
            stats["missing_example_id"] += 1
            continue

        if example_id not in gt_index:
            per_example[json_file.name] = {
                "example_id": example_id,
                "pred_answer_raw": raw_answer,
                "pred_answer_letter": None,
                "pred_answer_index": None,
                "gt_answer_index": None,
                "gt_answer_letter": None,
                "correct": None,
                "error": "example_id not found in ERQA index",
            }
            stats["missing_gt"] += 1
            continue

        gt = gt_index[example_id]

        num_choices = len(gt["choices"]) if gt["choices"] is not None else 4

        pred_letter = extract_option_letter(raw_answer, num_choices=num_choices)
        pred_index = None
        parse_mode = None

        if pred_letter is not None:
            pred_index = string.ascii_uppercase.index(pred_letter)
            parse_mode = "letter"
        elif gt["choices"] is not None:
            matched_idx = extract_choice_by_text(raw_answer, gt["choices"])
            if matched_idx is not None:
                pred_index = matched_idx
                pred_letter = string.ascii_uppercase[matched_idx]
                parse_mode = "choice_text"

        if pred_index is None:
            per_example[json_file.name] = {
                "example_id": example_id,
                "pred_answer_raw": raw_answer,
                "pred_answer_letter": None,
                "pred_answer_index": None,
                "gt_answer_index": gt["gt_answer_index"],
                "gt_answer_letter": gt["gt_answer_letter"],
                "gt_answer_text": gt["gt_answer_text"],
                "choices": gt["choices"],
                "correct": 0,
                "error": "Could not parse model answer as a valid option letter or choice text",
                "question_id": gt["question_id"],
                "question": gt["question"],
                "question_type": gt["question_type"],
                "row_index": gt["row_index"],
            }
            stats["unparsed_prediction"] += 1
            stats["evaluated"] += 1
            stats["incorrect"] += 1
            continue

        correct = 1 if pred_index == gt["gt_answer_index"] else 0

        per_example[json_file.name] = {
            "example_id": example_id,
            "pred_answer_raw": raw_answer,
            "pred_answer_letter": pred_letter,
            "pred_answer_index": pred_index,
            "pred_answer_text": (
                gt["choices"][pred_index]
                if gt["choices"] is not None and 0 <= pred_index < len(gt["choices"])
                else None
            ),
            "gt_answer_index": gt["gt_answer_index"],
            "gt_answer_letter": gt["gt_answer_letter"],
            "gt_answer_text": gt["gt_answer_text"],
            "choices": gt["choices"],
            "correct": correct,
            "parse_mode": parse_mode,
            "question_id": gt["question_id"],
            "question": gt["question"],
            "question_type": gt["question_type"],
            "row_index": gt["row_index"],
        }

        stats["evaluated"] += 1
        if correct == 1:
            stats["correct"] += 1
        else:
            stats["incorrect"] += 1

    accuracy = (
        stats["correct"] / stats["evaluated"]
        if stats["evaluated"] > 0
        else 0.0
    )

    output = {
        "summary": {
            "results_dir": str(results_dir),
            "dataset_name": dataset_name,
            "split": split,
            "num_json_files": len(json_files),
            "evaluated": stats["evaluated"],
            "correct": stats["correct"],
            "incorrect": stats["incorrect"],
            "accuracy": accuracy,
            "missing_example_id": stats["missing_example_id"],
            "missing_gt": stats["missing_gt"],
            "unparsed_prediction": stats["unparsed_prediction"],
            "read_error": stats["read_error"],
        },
        "per_example": per_example,
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    print(f"Saved evaluation to: {output_json_path}")
    print(f"Accuracy: {accuracy:.4f} ({stats['correct']}/{stats['evaluated']})")


if __name__ == "__main__":
    evaluate_result_folder(
        results_dir="30b_erqa_analysis",
        output_json_path="30b_erqa_evaluation_results.json",
        dataset_name="FlagEval/ERQA",
        split="test",
    )